# Lesson 6: One controlled emission session

**This notebook can produce a laser beam.** Use only in an attended,
operator-approved setup. It checks readiness, stores and verifies power, enables,
opens the safety shutter once, reads reported power, closes the shutter and
returns to standby.

Complete [lesson 4](04_pre_key_on_readiness.ipynb) first: warm-up must finish before
physical key-on. The operator must establish key ON / remote standby with the
shutter closed using the approved procedure. Turning the key can itself enable
emission; do not turn it merely to satisfy software. Confirm cooling, interlocks,
beam termination, suitable eyewear and a physical abort procedure.

Only byte-based `read()` and `write()` exchanges are used. Each waits for the
full reply and prompt. Raw replies and acknowledgments are saved unchanged;
byte payloads are compared explicitly, without encoding or decoding.

**After any error, stop.** A command may already have executed and the shutter
may remain open or the laser enabled. Use the physical abort procedure. There
are no retries or automatic cleanup writes. Disconnecting or interrupting the
kernel is not an emergency stop. Never use the safety shutter as an experimental
modulator. Keep one serial owner and restart the kernel after package updates.

## 1. Connection and approved power settings

Confirm port, model and baud with the operator. Enter the independently verified
no-active-fault reply as **bytes**, for example `b"SYSTEM OK"` only if verified for
this instrument. Do not infer it from fault history or an empty response.
Set `TARGET_W` and the site-approved `POWER_LIMIT_W` in watts; no default power
is supplied. Restart the kernel after package updates.

In [ ]:
PORT = "<PORT>"
MODEL = "<MODEL>"  # V2, V5 or V6
BAUDRATE = 19200
TIMEOUT_S = 2.0
ACTIVE_FAULT_CLEAR_REPLY = None  # Verified payload bytes, not a full wire reply
TARGET_W = None  # Operator-approved setpoint in watts
POWER_LIMIT_W = None  # Site-approved ceiling in watts

## 2. Validate settings before opening the port

The instruction is built directly with byte formatting: `b"P=%.4f"`.
Both the original target and the four-decimal transmitted value must be within
the ceiling. Construction sends nothing. Writes require explicit enablement.

In [ ]:
from math import isfinite

from coherent_verdi import VerdiController

if PORT == "<PORT>" or POWER_LIMIT_W is None or ACTIVE_FAULT_CLEAR_REPLY is None:
    raise ValueError("Fill in every setting in section 1 first.")
if (
    not isinstance(ACTIVE_FAULT_CLEAR_REPLY, bytes)
    or not ACTIVE_FAULT_CLEAR_REPLY
    or ACTIVE_FAULT_CLEAR_REPLY.strip(b" ") != ACTIVE_FAULT_CLEAR_REPLY
    or any(c < 32 or c > 126 for c in ACTIVE_FAULT_CLEAR_REPLY)
    or b"&" in ACTIVE_FAULT_CLEAR_REPLY
    or (ACTIVE_FAULT_CLEAR_REPLY.isdigit() and int(ACTIVE_FAULT_CLEAR_REPLY) > 0)
):
    raise ValueError("ACTIVE_FAULT_CLEAR_REPLY must be verified nonempty payload bytes.")
laser = VerdiController(
    PORT,
    model=MODEL,
    baudrate=BAUDRATE,
    timeout_s=TIMEOUT_S,
    prompt_enabled=True,
    allow_writes=True,
    power_limit_w=POWER_LIMIT_W,
)
if type(TARGET_W) not in (int, float) or not isfinite(TARGET_W):
    raise ValueError("TARGET_W must be a finite number.")
if not 0 <= TARGET_W <= laser.power_limit_w:
    raise ValueError("TARGET_W must be between 0 and POWER_LIMIT_W.")
target_bytes = b"%.4f" % TARGET_W
sent_w = float(target_bytes)
if not 0 <= sent_w <= laser.power_limit_w:
    raise ValueError("Rounded TARGET_W exceeds POWER_LIMIT_W.")
power_command = b"P=" + target_bytes
print("Will send:", repr(power_command))

## 3. Compare byte payloads explicitly

Raw replies stay unchanged in the notebook and transaction record. This small
helper checks the known terminal framing and removes a recognized prompt and
exact instruction echo **only for the comparisons below**. It never encodes or
decodes text and does not attempt to repair interleaved or corrupted replies.

In [ ]:
prompts = (b"Verdi>", {"V2": b"Verdi V-2>", "V5": b"Verdi V-5>", "V6": b"Verdi V-6>"}[MODEL])


def reply_payload(instruction, wire):
    for prompt in prompts:
        if wire.endswith(b"\r\n" + prompt):
            body = wire[: -(len(prompt) + 2)]
            break
    else:
        raise RuntimeError(f"Unexpected reply framing: {wire!r}")
    for prompt in prompts:
        if body.startswith(prompt):
            body = body[len(prompt) :].lstrip(b" ")
            break
    body = body.removeprefix(instruction).strip(b" ")
    if b"\r" in body or b"\n" in body:
        raise RuntimeError(f"Unexpected extra reply data: {wire!r}")
    return body

## 4. Open the serial port

Run this cell by itself and inspect the result before starting serial communication.
`connect()` opens the local serial port and returns `True` on success. Failures
raise an exception. It sends no instrument instructions. A successful open does
not establish that the laser is responding or ready.

Close any other serial owner first. If opening fails, stop here.

`override=True` closes an older controller connection to the same port in this
Python process before opening the new one. It cannot take ownership from another
kernel/application or a raw `ser` handle; release those explicitly. A failed
controller still requires a fresh controller object. Reopening does not establish
instrument synchronization or repair corrupted replies.

In [ ]:
connected = laser.connect(override=True)
print("Serial port open:", connected)

## 5. Start serial communication

Only after inspecting the connection result, run the following cells separately.
Buffer maintenance and instrument queries belong here, outside the connection
section. An explicit purge discards host queues; it does not reset the device's
parser. No delay or purge is claimed to prove synchronization. Each query waits
for its complete reply and prompt before another can be sent.

In [ ]:
laser.purge_buffers()
replies = []
acknowledgments = []


def read_checked(query):
    wire = laser.read(query)
    replies.append((query, wire))
    print(query, repr(wire))
    payload = wire
    if not payload or payload.startswith((b"RANGE ERROR:", b"Command Error:", b"Query Error:")):
        raise RuntimeError(f"Unexpected query reply: {wire!r}")
    return payload


def command_checked(instruction):
    wire = laser.write(instruction)
    acknowledgments.append((instruction, wire))
    print(instruction, "acknowledgment:", repr(wire))
    if reply_payload(instruction, wire) != b"":
        raise RuntimeError(f"Unexpected command acknowledgment: {wire!r}")
    return wire


version_reply = read_checked(b"?SV")

## 6. Check readiness

Require standby, closed shutter, the verified no-active-fault reply, key ON and
all four temperature servos LOCKED. Checks are sequential and cannot prevent
external state changes between requests; physical safeguards remain essential.
Readiness is rechecked after each operator confirmation and after storing power.

In [ ]:
def check_ready():
    state = {query: read_checked(query) for query in (b"?L", b"?S", b"?F", b"?K")}
    if state[b"?L"] != b"0" or state[b"?S"] != b"0" or state[b"?F"] != ACTIVE_FAULT_CLEAR_REPLY:
        raise RuntimeError("Require STANDBY, shutter closed and no active faults.")
    if state[b"?K"] != b"1":
        raise RuntimeError("Require the operator's key-ON / standby configuration.")
    for query in (b"?D1SS", b"?LBOSS", b"?ESS", b"?VSS"):
        if read_checked(query) != b"1":
            raise RuntimeError(f"Temperature servo {query!r} is not LOCKED; do not enable.")


check_ready()
print("Reported readiness: standby, shutter closed, no faults, key ON, servos locked.")

## 7. Preserve fault history before enabling

`L=1` resets faults and clears history. Save the entire raw history reply first.
A clear history alone does not establish readiness. Never enable as a fault-reset
or recovery action.

In [ ]:
history_before_enable = laser.read(b"?FH")
print("Fault history before enable:", repr(history_before_enable))
history_payload = history_before_enable
if not history_payload or history_payload.startswith(
    (b"RANGE ERROR:", b"Command Error:", b"Query Error:")
):
    raise RuntimeError(f"Unexpected fault history reply: {history_before_enable!r}")

## 8. Set and verify power while still in standby

Type `SET` to store the approved setpoint. Recheck readiness immediately before
the write. Require a matching four-decimal `?SP` byte payload, then check readiness
again. A mismatch or failed check stops before enabling.

In [ ]:
if input("To store the setpoint, type SET: ").strip() != "SET":
    raise RuntimeError("Cancelled. No settings were changed.")
check_ready()
command_checked(power_command)
if read_checked(b"?SP") != target_bytes:
    raise RuntimeError("Setpoint readback mismatch; do not enable.")
check_ready()

## 9. Run the emission sequence

Type `RUN` only after physical preparation is complete. The sequence is one cell:
there is no keyboard prompt or deliberate dwell while the laser is enabled.

| Step | Byte command/query | Verify before proceeding |
| --- | --- | --- |
| Enable | `b"L=1"` | ON, no active faults, shutter closed |
| Open safety shutter | `b"S=1"` | Open, ON, no active faults |
| Read reported power | `b"?P"` | Preserve raw reading |
| Close safety shutter | `b"S=0"` | Closed |
| Return to standby | `b"L=0"` | STANDBY, shutter closed, no faults, stored setpoint unchanged |

Each command consumes its acknowledgment and prompt before readback begins.
The power sample is immediate; it establishes neither settling nor calibration.
On failure, execution stops with no blind cleanup or replay.

In [ ]:
if input("To perform the emission sequence, type RUN: ").strip() != "RUN":
    raise RuntimeError("Cancelled. The laser was not enabled.")
check_ready()
if read_checked(b"?SP") != target_bytes:
    raise RuntimeError("Stored setpoint differs from TARGET_W; do not enable.")

command_checked(b"L=1")
if (
    read_checked(b"?L") != b"1"
    or read_checked(b"?F") != ACTIVE_FAULT_CLEAR_REPLY
    or read_checked(b"?S") != b"0"
):
    raise RuntimeError("ON with no faults and a closed shutter was not confirmed.")

command_checked(b"S=1")
if (
    read_checked(b"?S") != b"1"
    or read_checked(b"?L") != b"1"
    or read_checked(b"?F") != ACTIVE_FAULT_CLEAR_REPLY
):
    raise RuntimeError("Open shutter, ON and no faults were not confirmed.")
power_reply = laser.read(b"?P")
print("Reported power while open (raw):", repr(power_reply))
power_payload = power_reply
if not power_payload or any(marker in power_payload for marker in (b"Error:", b"ERROR:")):
    raise RuntimeError(f"Unexpected power reply: {power_reply!r}")

command_checked(b"S=0")
if read_checked(b"?S") != b"0":
    raise RuntimeError("Shutter closure was not confirmed.")

command_checked(b"L=0")
final_payloads = {query: read_checked(query) for query in (b"?L", b"?S", b"?F", b"?SP")}
if (
    final_payloads[b"?L"] != b"0"
    or final_payloads[b"?S"] != b"0"
    or final_payloads[b"?F"] != ACTIVE_FAULT_CLEAR_REPLY
    or final_payloads[b"?SP"] != target_bytes
):
    raise RuntimeError("Final standby, shutter closure, fault state or setpoint was not confirmed.")
print("Shutter closed and STANDBY reported; setpoint remains stored.")

## 10. Inspect an error

Inspect the last transaction and retained `replies`/`acknowledgments` without
sending anything. Use the physical abort procedure if needed before releasing
the port. Do not retry an uncertain write or send automatic cleanup commands.

In [ ]:
print(laser.last_transaction)

## 11. Disconnect

Release the serial port without changing laser state. Normal completion above
has verified shutter closure and standby. Disconnect does not perform those
operations after a failure. Standby leaves temperature servos powered; full
shutdown follows the manual's separate heater and cooling procedure.

In [ ]:
laser.disconnect()